In [ ]:
import json
from pathlib import Path

import numpy as np
from ising.model import FitMethod, UpdateMethod

from climate_attitudes.dataset import Dataset
from climate_attitudes.settings import Config
from climate_attitudes.utils import calculate_stationary_distribution
from climate_attitudes.visualisation import configure_mpl
from ising import Ising

configure_mpl(Path("../fonts/"))

np.set_printoptions(linewidth=200)

RANDOM_SEED = 202607161539
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
config = Config(_env_file="../.env")
dataset = Dataset.load(
    config,
    name="reduced_no_imputation",
    with_imputation=False,
    verbose=False,
)

sigma_path = Path("../reports/thesis/results/data/methods/binarisation_sigma.json")
lambda_path = Path(
    "../reports/thesis/results/data/model_fit/optimised_regularisation.json"
)

with sigma_path.open("r") as f:
    sigma = json.load(f)["sigma"]

with lambda_path.open("r") as f:
    λ = json.load(f)["ising"]["full"]

# Fit model
_, _, P, *_ = dataset.indices_to_numpy(
    kind="time-series",
    binarise=True,
    scale=sigma,
    seed=rng,
    binarisation_dist="gaussian",
)
model = Ising.fit(
    y=P,
    optim_method=FitMethod.TIME_SERIES,
    update_method=UpdateMethod.SYNCHRONOUS,
    rng=rng,
    adj=None,
    self_loops=True,
    w=λ,
)

Extract transition probabilities

In [ ]:
N = model.size
Y0 = (2 * ((np.arange(1 << N)[:, None] >> np.arange(N)) & 1) - 1).astype(np.float64)
X = np.ones(Y0.shape[0])

In [ ]:
heff = model.parallel_glauber_theta_batch(Y0, X, model.h, model.j, model.adj)

In [ ]:
sum_log2cosh_heff = np.log(2 * np.cosh(heff)).sum(axis=-1)

In [ ]:
s_dot_heff = heff @ Y0.T

In [ ]:
p_transition = np.exp(s_dot_heff - sum_log2cosh_heff)

In [ ]:
mu = calculate_stationary_distribution(p_transition)

In [ ]:
Y0[mu > 0.008]

1. Believe, support, not worried.
2. Skeptic, oppose, worried.

Calculate probability distribution over configurations given the first wave of data

In [ ]:
empirical_distribution = np.empty_like(mu)

In [ ]:
for state_idx, s in enumerate(Y0):
    empirical_distribution[state_idx] = (
        1
        / P.shape[0]
        * np.exp((np.log((1 - s) / 2 * (1 - P) + (1 + s) / 2 * P)).sum(axis=-1)).sum()
        / 2
    )

In [ ]:
empirical_distribution.sum()

In [ ]:
np.abs(empirical_distribution - mu).mean()